# STA6130 Week 8 Group Assignment: Forecasting as Regression and Feature Engineering
**Dataset:** Store Item Demand Forecasting Challenge (Kaggle), daily unit sales, 1 Jan 2013 to 31 Dec 2017  
**Competition:** https://www.kaggle.com/c/demand-forecasting-kernels-only  (data: `train.csv` with columns `date, store, item, sales`)  

**Mapping to the assignment:** sections 1 to 10 below follow the ten required report items, so each output can be pasted straight into the matching part of the report.

**Running the notebook:** use **Run All** after starting/restarting the kernel, or use **Run Above** before running an individual downstream cell. Notebook cells share variables in the kernel; a cell that uses results from earlier cells cannot run by itself in a fresh kernel. No terminal commands are needed for normal notebook execution.

In [ ]:
import os, glob, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import skew
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error

# Analysis settings: adjust these values here before running the notebook.
STORE = 1
ITEM = 1
H = 1  # one-day-ahead forecast horizon
TRAIN_FRAC = 0.8  # chronological training fraction

## 1. Describe the forecasting problem

In [ ]:
def find_train_csv():
    # 1) a copy next to the notebook, 2) the Kaggle notebook input folder, 3) kagglehub download
    for path in ["train.csv", "/kaggle/input/demand-forecasting-kernels-only/train.csv"]:
        if os.path.exists(path):
            return path
    try:
        import kagglehub
        folder = kagglehub.competition_download("demand-forecasting-kernels-only")
        hits = glob.glob(os.path.join(folder, "**", "train.csv"), recursive=True)
        if hits:
            return hits[0]
    except Exception as e:
        print("kagglehub download failed:", e)
    raise FileNotFoundError(
        "train.csv not found. Join the competition, download the data from"
        )

raw = pd.read_csv(find_train_csv(), parse_dates=["date"])
print("Full file:", raw.shape, "| stores:", raw["store"].nunique(), "| items:", raw["item"].nunique())
print("Period:", raw["date"].min().date(), "to", raw["date"].max().date())

# One store-item series (local model).
s = (raw[(raw["store"] == STORE) & (raw["item"] == ITEM)]
     .sort_values("date").set_index("date")["sales"].astype(float).asfreq("D"))
assert s.isna().sum() == 0, "missing days found: impute carefully (never with future values)"
print(f"\nSeries: store {STORE}, item {ITEM}: {len(s)} daily observations, {s.index.min().date()} to {s.index.max().date()}")
print(s.describe().round(2))

In [ ]:
# Exploratory plots: level/trend, weekly pattern, yearly pattern
fig, axes = plt.subplots(3, 1, figsize=(10, 10))
s.plot(ax=axes[0], lw=0.6, title=f"Daily sales, store {STORE} item {ITEM}")
s.groupby(s.index.dayofweek).mean().plot(kind="bar", ax=axes[1], title="Average sales by day of week (0 = Monday)")
s.groupby(s.index.month).mean().plot(kind="bar", ax=axes[2], title="Average sales by month")
plt.tight_layout(); plt.show()

# Autocorrelation at candidate lags (justifies which lags to engineer)
for k in [1, 2, 3, 7, 14, 21, 28, 364]:
    print(f"autocorrelation at lag {k:3d}: {s.autocorr(lag=k):.3f}")

## 2 and 3. Convert the series into a supervised regression table and create features

Every row is one **day to be predicted** (its `target` is that day's sales). All inputs are built only from information known `H` days earlier, so row *T* never sees the sales of day *T*.

| Category | Features | Why they help |
|---|---|---|
| Lag | `lag_k` for k = 1, 2, 3, 7, 14, 28 (only k >= H) | Persistence and weekly memory |
| Rolling (shifted) | mean, std, min, max over 7 and 28 days | Recent level, volatility and range |
| EWMA (shifted) | exponentially weighted average | Recent level with more weight on newer days |
| Calendar | day of week, weekend flag, month, quarter, week of year | Weekly and annual patterns |
| Trend | `time_index` | Long-run growth |
| Seasonal (Fourier) | sin/cos terms with periods 7 and 365.25 | Smooth weekly and yearly cycles |

In [ ]:
def build_features(y: pd.Series, H: int = 1) -> pd.DataFrame:
    df = pd.DataFrame({"target": y})

    # --- Lag features (k days before the day being predicted; must satisfy k >= H to be known in advance)
    for k in [1, 2, 3, 7, 14, 28]:
        if k >= H:
            df[f"lag_{k}"] = y.shift(k)

    # --- Rolling features: SHIFT FIRST so the window ends at the last known day
    known = y.shift(H)
    for w in [7, 28]:
        r = known.rolling(w)
        df[f"roll_mean_{w}"] = r.mean()
        df[f"roll_std_{w}"]  = r.std()
        df[f"roll_min_{w}"]  = r.min()
        df[f"roll_max_{w}"]  = r.max()
    df["ewma_alpha03"] = known.ewm(alpha=0.3, adjust=False).mean()

    # --- Calendar features (known in advance for any future date)
    idx = df.index
    df["day_of_week"]  = idx.dayofweek
    df["is_weekend"]   = (idx.dayofweek >= 5).astype(int)
    df["month"]        = idx.month
    df["quarter"]      = idx.quarter
    df["week_of_year"] = idx.isocalendar().week.astype(int).values

    # --- Trend feature
    df["time_index"] = np.arange(len(df))

    # --- Fourier (seasonal) features
    t = df["time_index"].values
    for k in (1, 2):
        df[f"sin_week_{k}"] = np.sin(2 * np.pi * k * t / 7)
        df[f"cos_week_{k}"] = np.cos(2 * np.pi * k * t / 7)
    for k in (1, 2, 3):
        df[f"sin_year_{k}"] = np.sin(2 * np.pi * k * t / 365.25)
        df[f"cos_year_{k}"] = np.cos(2 * np.pi * k * t / 365.25)
    return df

features_all = build_features(s, H)
model_df = features_all.dropna().copy()          # rows without full lag/rolling history are dropped
feature_cols = [c for c in model_df.columns if c != "target"]
print(f"Rows before dropping: {len(features_all)} | after: {len(model_df)} | features: {len(feature_cols)}")
model_df[["target"] + feature_cols].head(8).round(2)

# 4. Avoid Data Leakage
#### Automated leakage test: change the value we are trying to predict on a given day (add 1000 units).
#### If any feature in THAT row changes, the feature is using information it should not have.

In [ ]:
def leakage_check(y, H=1, n_checks=8, seed=0):
    rng = np.random.default_rng(seed)
    base = build_features(y, H).drop(columns=["target"])
    for i in rng.choice(np.arange(60, len(y)), n_checks, replace=False):
        y2 = y.copy(); y2.iloc[i] += 1000
        alt = build_features(y2, H).drop(columns=["target"])
        assert np.allclose(base.iloc[i].values.astype(float), alt.iloc[i].values.astype(float), equal_nan=True), \
            f"LEAKAGE: features on row {i} change when its own target changes"
    print(f"Leakage check passed on {n_checks} random days: no feature uses the target day's own sales.")

leakage_check(s, H)

# For contrast, the WRONG way (window includes the current day) is detected:
wrong = s.rolling(7).mean()
s2 = s.copy(); s2.iloc[500] += 1000
print("Wrong rolling mean changes when today's sales change:", wrong.iloc[500] != s2.rolling(7).mean().iloc[500])

Leakage steps taken (use in the report): (1) every rolling/EWMA feature is **shifted by H before** it is computed; (2) lag k is only used when **k >= H**; (3) calendar, trend and Fourier features depend only on the date, which is known in advance; (4) **no external variables** are used (none are provided), so nothing unknown at forecast time enters; (5) scaling is fitted **inside a pipeline on the training data only**; (6) the split is **chronological**, never random; and (7) an **automated test** proves no feature reacts to its own target.

## 5. Chronological split

In [ ]:
split = int(len(model_df) * TRAIN_FRAC)
train, test = model_df.iloc[:split], model_df.iloc[split:]
X_train, y_train = train[feature_cols], train["target"]
X_test,  y_test  = test[feature_cols],  test["target"]
print(f"Train: {train.index.min().date()} to {train.index.max().date()}  ({len(train)} days)")
print(f"Test : {test.index.min().date()} to {test.index.max().date()}  ({len(test)} days)")